# IDX Pump-and-Dump Detection — Pipeline

**Research:** Detecting pump-and-dump manipulation on the Indonesia Stock Exchange (IDX) using ML and DL.  
**Period:** 2021–2026 | **Labels:** IDX UMA announcements | **Primary metric:** MCC + full confusion matrix

This notebook covers the full pipeline:
1. Setup
2. Data collection (yfinance + UMA labels)
3. Exploratory Data Analysis
4. Feature engineering
5. Labeling
6. Preprocessing & train/test split
7. Classical ML models (Logistic Regression, Random Forest, LightGBM)
8. Deep Learning models (BiLSTM, CNN-LSTM, Transformer — Keras/TF)
9. Save all results

---
## Section 1 — Setup

In [ ]:
import os, warnings, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf
import ta
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (confusion_matrix, matthews_corrcoef,
                             precision_score, recall_score, f1_score,
                             average_precision_score, balanced_accuracy_score)
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Constants ────────────────────────────────────────────────────────────────
START         = '2021-01-01'
END           = '2026-12-31'
TEST_CUTOFF   = '2025-01-01'   # time-based split: train < this, test >= this
EVENT_WINDOW  = (-5, 5)        # days around UMA announcement date
WINDOW_SIZE   = 20             # rolling feature window (trading days)
BATCH_SIZE    = 50             # tickers per yfinance download call

DATA_DIR      = 'data/raw/ohlcv_daily'
PROC_DIR      = 'data/processed'
MODEL_DIR     = 'models'
RESULTS_DIR   = 'results'

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete. TF version:', tf.__version__)

---
## Section 2 — Data Collection

### 2a — UMA label list

> **Ground-truth note (§7.2 of research doc):** A UMA announcement signals *suspicion* of unusual market activity — the IDX explicitly states it does **not** necessarily indicate a rule violation. Labels are therefore weak/noisy and treated as such throughout this study.

In [ ]:
uma_df = pd.read_csv('data/raw/uma_events.csv', parse_dates=['uma_date'])
uma_df['ticker'] = uma_df['ticker'].str.upper()
print(f'UMA events loaded: {len(uma_df)} records spanning '
      f"{uma_df['uma_date'].min().date()} – {uma_df['uma_date'].max().date()}")
uma_df.head()

### 2b — IDX ticker universe

We use the tickers present in `uma_events.csv` plus the IHSG benchmark (`^JKSE`).  
For a full production run, replace `tickers` with the complete IDX common-stock universe stored in `data/raw/idx_tickers.txt` (one `.JK` ticker per line).

In [ ]:
# ── Load or derive ticker list ────────────────────────────────────────────────
ticker_file = 'data/raw/idx_tickers.txt'
if os.path.exists(ticker_file):
    with open(ticker_file) as f:
        tickers = [t.strip().upper() for t in f if t.strip()]
else:
    # Fall back to UMA tickers only (for development/testing)
    tickers = sorted(uma_df['ticker'].unique().tolist())
    print('idx_tickers.txt not found — using UMA tickers only (development mode)')

print(f'Ticker universe: {len(tickers)} stocks')
print(tickers[:10], '...')

In [ ]:
def download_ohlcv_batch(tickers_batch, start, end):
    """Download OHLCV for a batch of tickers; return dict of DataFrames."""
    result = {}
    try:
        raw = yf.download(
            tickers_batch, start=start, end=end,
            group_by='ticker', auto_adjust=True, progress=False
        )
        if len(tickers_batch) == 1:
            # Single-ticker download has flat columns
            ticker = tickers_batch[0]
            raw.columns = pd.MultiIndex.from_product([[ticker], raw.columns])
        for t in tickers_batch:
            if t in raw.columns.get_level_values(0):
                df = raw[t].dropna(how='all')
                if not df.empty:
                    result[t] = df
    except Exception as e:
        print(f'  Batch error: {e}')
    return result


# ── Download in batches ───────────────────────────────────────────────────────
all_ohlcv = {}
failed    = []
batches   = [tickers[i:i+BATCH_SIZE] for i in range(0, len(tickers), BATCH_SIZE)]

for i, batch in enumerate(batches):
    print(f'Downloading batch {i+1}/{len(batches)} ({len(batch)} tickers)...')
    data = download_ohlcv_batch(batch, START, END)
    for t, df in data.items():
        # Persist to disk
        path = f'{DATA_DIR}/{t}.csv'
        df.to_csv(path)
        all_ohlcv[t] = df
    batch_failed = [t for t in batch if t not in data]
    failed.extend(batch_failed)

with open('data/raw/failed_tickers.txt', 'w') as f:
    f.write('\n'.join(failed))

print(f'\nDownloaded: {len(all_ohlcv)} tickers | Failed: {len(failed)}')

In [ ]:
# ── 2c — IHSG benchmark ───────────────────────────────────────────────────────
ihsg = yf.download('^JKSE', start=START, end=END, auto_adjust=True, progress=False)
# yfinance >=0.2 returns MultiIndex columns even for single tickers — squeeze to Series
ihsg_close = ihsg['Close'].squeeze()
ihsg_returns = ihsg_close.pct_change().rename('ihsg_return')
print(f'IHSG loaded: {len(ihsg)} trading days')

---
## Section 3 — Exploratory Data Analysis

In [ ]:
# ── 3a — UMA event timeline ───────────────────────────────────────────────────
uma_df['year'] = uma_df['uma_date'].dt.year
fig, ax = plt.subplots(figsize=(8, 4))
uma_df['year'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('UMA Announcements per Year')
ax.set_xlabel('Year'); ax.set_ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3b — Price move and volume around UMA events ──────────────────────────────
event_stats = []
for _, row in uma_df.iterrows():
    t = row['ticker']
    if t not in all_ohlcv:
        continue
    df = all_ohlcv[t].copy()
    df.index = pd.to_datetime(df.index)
    event_date = row['uma_date']
    # 5-day window before announcement
    mask = (df.index >= event_date - pd.Timedelta(days=7)) & (df.index <= event_date)
    window = df[mask]
    if len(window) < 2:
        continue
    price_move = (window['Close'].iloc[-1] / window['Close'].iloc[0] - 1) * 100
    avg_vol   = df['Volume'].rolling(20).mean().reindex(window.index).iloc[-1]
    vol_ratio = window['Volume'].iloc[-1] / avg_vol if avg_vol > 0 else np.nan
    event_stats.append({'ticker': t, 'price_move_pct': price_move, 'volume_ratio': vol_ratio})

stats_df = pd.DataFrame(event_stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(stats_df['price_move_pct'].dropna(), bins=20, color='coral', edgecolor='white')
axes[0].set_title('Price Move % (5d before UMA)')
axes[0].set_xlabel('Price Change (%)')
axes[1].hist(stats_df['volume_ratio'].dropna().clip(0, 10), bins=20, color='teal', edgecolor='white')
axes[1].set_title('Volume Ratio vs 20d Avg at UMA')
axes[1].set_xlabel('Volume Ratio (×)')
plt.tight_layout(); plt.show()
print(stats_df.describe())

In [ ]:
# ── 3c — Average OHLCV behavior H-10 to H+10 around UMA events ───────────────
relative_prices = []
for _, row in uma_df.iterrows():
    t = row['ticker']
    if t not in all_ohlcv:
        continue
    df = all_ohlcv[t].copy()
    df.index = pd.to_datetime(df.index)
    event_date = row['uma_date']
    window = df[(df.index >= event_date - pd.Timedelta(days=15))
                & (df.index <= event_date + pd.Timedelta(days=15))].copy()
    if len(window) < 10:
        continue
    # Normalize price to 1.0 on the day nearest the event
    idx_event = window.index.get_indexer([event_date], method='nearest')[0]
    base_price = window['Close'].iloc[idx_event]
    window['norm_close'] = window['Close'] / base_price
    window['rel_day'] = range(-idx_event, len(window) - idx_event)
    relative_prices.append(window[['rel_day', 'norm_close']])

if relative_prices:
    all_rel = pd.concat(relative_prices)
    agg = all_rel.groupby('rel_day')['norm_close'].agg(['mean', 'std']).reset_index()
    agg = agg[agg['rel_day'].between(-10, 10)]
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(agg['rel_day'], agg['mean'], marker='o', color='steelblue', label='Mean normalized close')
    ax.fill_between(agg['rel_day'],
                    agg['mean'] - agg['std'],
                    agg['mean'] + agg['std'],
                    alpha=0.25, color='steelblue', label='±1 std')
    ax.axvline(0, color='red', linestyle='--', label='UMA announcement day')
    ax.set_title('Normalized Close Price Around UMA Event (H-10 to H+10)')
    ax.set_xlabel('Trading days relative to UMA date')
    ax.set_ylabel('Normalized price (1.0 = event day)')
    ax.legend()
    plt.tight_layout(); plt.show()

---
## Section 4 — Feature Engineering

We build a **rolling 20-day feature vector** for each (ticker, window_end_date) observation. The final row of each 20-day window is the observation date. Features capture returns, volume, volatility, momentum, and temporal signals.

In [ ]:
def compute_features(df: pd.DataFrame, ihsg_returns: pd.Series) -> pd.DataFrame:
    """Compute rolling features for a single ticker. Returns a DataFrame indexed by date."""
    df = df.copy()
    df.index = pd.to_datetime(df.index)

    # ── Returns ──────────────────────────────────────────────────────────────
    df['return_1d']  = df['Close'].pct_change()
    df['return_5d']  = df['Close'].pct_change(5)
    df['return_20d'] = df['Close'].pct_change(20)
    ihsg_aligned     = ihsg_returns.reindex(df.index).fillna(0)
    df['car']        = df['return_5d'] - ihsg_aligned.rolling(5).sum()

    # ── Volume ───────────────────────────────────────────────────────────────
    vol_avg          = df['Volume'].rolling(20).mean()
    df['tva_ratio']  = df['Volume'] / vol_avg.replace(0, np.nan)
    vol_std          = df['Volume'].rolling(20).std()
    df['vol_spike']  = ((df['Volume'] - vol_avg) / vol_std.replace(0, np.nan)).fillna(0)
    close_avg        = df['Close'].rolling(20).mean().replace(0, np.nan)
    df['turnover']   = df['Volume'] / close_avg

    # ── Volatility ───────────────────────────────────────────────────────────
    df['vol_5d']     = df['return_1d'].rolling(5).std()
    df['hl_ratio']   = (df['High'] - df['Low']) / df['Close'].replace(0, np.nan)
    df['atr_20d']    = ta.volatility.AverageTrueRange(
                           df['High'], df['Low'], df['Close'], window=20
                       ).average_true_range()

    # ── Momentum ─────────────────────────────────────────────────────────────
    df['rsi_14']     = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
    ma5              = df['Close'].rolling(5).mean()
    ma20             = df['Close'].rolling(20).mean().replace(0, np.nan)
    df['ma_ratio']   = ma5 / ma20
    df['price_accel']= df['return_1d'].diff()

    # ── Temporal ─────────────────────────────────────────────────────────────
    df['dow_sin']    = np.sin(2 * np.pi * df.index.dayofweek / 5)
    df['dow_cos']    = np.cos(2 * np.pi * df.index.dayofweek / 5)

    feature_cols = [
        'return_1d', 'return_5d', 'return_20d', 'car',
        'tva_ratio', 'vol_spike', 'turnover',
        'vol_5d', 'hl_ratio', 'atr_20d',
        'rsi_14', 'ma_ratio', 'price_accel',
        'dow_sin', 'dow_cos'
    ]
    return df[feature_cols]


FEATURE_COLS = [
    'return_1d', 'return_5d', 'return_20d', 'car',
    'tva_ratio', 'vol_spike', 'turnover',
    'vol_5d', 'hl_ratio', 'atr_20d',
    'rsi_14', 'ma_ratio', 'price_accel',
    'dow_sin', 'dow_cos'
]

print(f'Feature set: {len(FEATURE_COLS)} features per time step')

In [ ]:
# ── Build flat feature matrix (one row per window_end_date) ───────────────────
# For ML models: use only the last (most recent) row of each 20-day window.
# For DL models: the full 20-day sequence is stored separately in Section 6.

records = []
for ticker, df in all_ohlcv.items():
    feat = compute_features(df, ihsg_returns)
    feat = feat.dropna()
    for date, row in feat.iterrows():
        rec = {'ticker': ticker, 'window_end_date': date}
        rec.update(row.to_dict())
        records.append(rec)

features_df = pd.DataFrame(records)
features_df['window_end_date'] = pd.to_datetime(features_df['window_end_date'])
print(f'Feature matrix: {features_df.shape[0]:,} rows × {features_df.shape[1]} columns')
features_df.head()

---
## Section 5 — Labeling

**Positive class (label=1):** a (ticker, date) pair is positive if that date falls within the event window `[uma_date + EVENT_WINDOW[0], uma_date + EVENT_WINDOW[1]]` for that ticker.

**Heuristic expansion:** windows with ≥25% price move AND volume >5× 20d avg, but *no* UMA event, are flagged in `label_heuristic` for ablation analysis.

In [ ]:
# ── Primary UMA labels ────────────────────────────────────────────────────────
features_df['label'] = 0

for _, uma_row in uma_df.iterrows():
    t  = uma_row['ticker']
    d  = uma_row['uma_date']
    lo = d + pd.Timedelta(days=EVENT_WINDOW[0])
    hi = d + pd.Timedelta(days=EVENT_WINDOW[1])
    mask = (
        (features_df['ticker'] == t) &
        (features_df['window_end_date'] >= lo) &
        (features_df['window_end_date'] <= hi)
    )
    features_df.loc[mask, 'label'] = 1

# ── Heuristic expansion labels ────────────────────────────────────────────────
features_df['label_heuristic'] = features_df['label'].copy()
heuristic_mask = (
    (features_df['label'] == 0) &
    (features_df['return_5d'] >= 0.25) &
    (features_df['tva_ratio'] >= 5.0)
)
features_df.loc[heuristic_mask, 'label_heuristic'] = 1

n_pos = features_df['label'].sum()
n_tot = len(features_df)
print(f'Class balance (primary labels):')
print(f'  Positive (P&D): {n_pos:,} ({n_pos/n_tot*100:.3f}%)')
print(f'  Negative:       {n_tot-n_pos:,} ({(n_tot-n_pos)/n_tot*100:.3f}%)')
print(f'\nHeuristic positive (incl. expansion): '
      f"{features_df['label_heuristic'].sum():,}")

In [ ]:
# ── Class imbalance visualization ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = features_df['label'].value_counts().sort_index()
counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Class Distribution (Primary Labels)')
axes[0].set_xticklabels(['Legitimate (0)', 'P&D (1)'], rotation=0)
axes[0].set_ylabel('Count')

axes[1].pie(counts.values, labels=['Legitimate', 'P&D'],
            colors=['steelblue', 'coral'], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Class Distribution (%)')

plt.tight_layout(); plt.show()

In [ ]:
# ── Save feature matrix ───────────────────────────────────────────────────────
features_df.to_csv(f'{PROC_DIR}/features.csv', index=False)
print(f'Saved: {PROC_DIR}/features.csv  ({features_df.shape[0]:,} rows)')

---
## Section 6 — Preprocessing & Train/Test Split

**Critical design choices:**
- **Time-based split** (not random): train = 2021–2024, test = 2025–2026. Random splits leak future data and hide concept drift.
- **SMOTE on training fold only.** Oversampling the test set inflates recall artificially — the single most common fatal error in imbalanced classification studies.
- **Scaler fit on train only**, then applied to both.

In [ ]:
features_df = pd.read_csv(f'{PROC_DIR}/features.csv', parse_dates=['window_end_date'])

# ── Time-based split ──────────────────────────────────────────────────────────
train_mask = features_df['window_end_date'] < TEST_CUTOFF
test_mask  = features_df['window_end_date'] >= TEST_CUTOFF

X_train_raw = features_df.loc[train_mask, FEATURE_COLS].values
y_train      = features_df.loc[train_mask, 'label'].values
X_test_raw   = features_df.loc[test_mask,  FEATURE_COLS].values
y_test       = features_df.loc[test_mask,  'label'].values

print(f'Train: {X_train_raw.shape[0]:,} samples | '
      f'Positive: {y_train.sum():,} ({y_train.mean()*100:.3f}%)')
print(f'Test:  {X_test_raw.shape[0]:,} samples  | '
      f'Positive: {y_test.sum():,} ({y_test.mean()*100:.3f}%)')

In [ ]:
# ── StandardScaler (fit on train only) ───────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)
joblib.dump(scaler, f'{MODEL_DIR}/scaler.pkl')

# ── SMOTE on training fold only ───────────────────────────────────────────────
smote = SMOTE(k_neighbors=5, random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

imbalance_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f'Imbalance ratio (neg/pos): {imbalance_ratio:.1f}x')
print(f'After SMOTE — train shape: {X_train_sm.shape} | '
      f'Positive: {y_train_sm.sum():,} ({y_train_sm.mean()*100:.1f}%)')

# ── Save processed splits ─────────────────────────────────────────────────────
pd.DataFrame(X_train_scaled, columns=FEATURE_COLS).to_csv(f'{PROC_DIR}/X_train.csv', index=False)
pd.DataFrame(X_test_scaled,  columns=FEATURE_COLS).to_csv(f'{PROC_DIR}/X_test.csv',  index=False)
pd.Series(y_train, name='label').to_csv(f'{PROC_DIR}/y_train.csv', index=False)
pd.Series(y_test,  name='label').to_csv(f'{PROC_DIR}/y_test.csv',  index=False)

print('Preprocessing complete.')

In [ ]:
def build_dl_sequences(features_df, ohlcv_dict, ihsg_returns, split_cutoff, window_size=20):
    """Build (samples, window_size, n_features) arrays for DL models.

    Each sample is a window_size-length sequence of daily feature vectors,
    ending on window_end_date. Label = last day's label.
    """
    X_seq, y_seq, dates_seq = [], [], []

    for ticker, grp in features_df.groupby('ticker'):
        if ticker not in ohlcv_dict:
            continue
        df = ohlcv_dict[ticker]
        feat = compute_features(df, ihsg_returns).dropna()
        feat = feat.sort_index()
        labels = features_df[features_df['ticker'] == ticker].set_index('window_end_date')['label']

        for i in range(window_size, len(feat)):
            end_date = feat.index[i]
            seq = feat.iloc[i - window_size: i][FEATURE_COLS].values
            if seq.shape != (window_size, len(FEATURE_COLS)):
                continue
            if end_date not in labels.index:
                continue
            X_seq.append(seq)
            y_seq.append(labels[end_date])
            dates_seq.append(end_date)

    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.int32)
    dates_seq = np.array(dates_seq)

    train_mask = dates_seq < pd.Timestamp(split_cutoff)
    test_mask  = dates_seq >= pd.Timestamp(split_cutoff)
    return (X_seq[train_mask], y_seq[train_mask],
            X_seq[test_mask],  y_seq[test_mask])


print('Building DL sequences (may take a few minutes)...')
X_dl_train_raw, y_dl_train, X_dl_test_raw, y_dl_test = build_dl_sequences(
    features_df, all_ohlcv, ihsg_returns, TEST_CUTOFF, WINDOW_SIZE
)

# Scale per-feature across the sequence axis using already-fitted scaler
n_train, T, F = X_dl_train_raw.shape
X_dl_train = scaler.transform(X_dl_train_raw.reshape(-1, F)).reshape(n_train, T, F)
n_test = X_dl_test_raw.shape[0]
X_dl_test  = scaler.transform(X_dl_test_raw.reshape(-1, F)).reshape(n_test, T, F)

class_weight_dl = {0: 1.0, 1: float(imbalance_ratio)}

print(f'DL train: {X_dl_train.shape} | Positive: {y_dl_train.sum():,}')
print(f'DL test:  {X_dl_test.shape}  | Positive: {y_dl_test.sum():,}')

---
## Section 7 — Classical ML Models

All three models are trained on the **SMOTE-augmented** training set and evaluated on the **original (un-augmented)** test set.

In [ ]:
results = {}  # Stores y_pred and y_proba for all models

def evaluate_and_store(name, y_true, y_pred, y_proba):
    results[name] = {'y_pred': y_pred, 'y_proba': y_proba}
    mcc = matthews_corrcoef(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    print(f'[{name}] MCC={mcc:.4f} | F1={f1:.4f} | '
          f"Recall={recall_score(y_true, y_pred, zero_division=0):.4f} | "
          f"Precision={precision_score(y_true, y_pred, zero_division=0):.4f}")

In [ ]:
# ── Logistic Regression (baseline) ───────────────────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr.fit(X_train_sm, y_train_sm)
joblib.dump(lr, f'{MODEL_DIR}/logistic_regression.pkl')

y_pred_lr   = lr.predict(X_test_scaled)
y_proba_lr  = lr.predict_proba(X_test_scaled)[:, 1]
evaluate_and_store('Logistic Regression', y_test, y_pred_lr, y_proba_lr)

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
rf.fit(X_train_sm, y_train_sm)
joblib.dump(rf, f'{MODEL_DIR}/random_forest.pkl')

y_pred_rf   = rf.predict(X_test_scaled)
y_proba_rf  = rf.predict_proba(X_test_scaled)[:, 1]
evaluate_and_store('Random Forest', y_test, y_pred_rf, y_proba_rf)

In [ ]:
# ── LightGBM ─────────────────────────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05,
    scale_pos_weight=imbalance_ratio,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm.fit(X_train_sm, y_train_sm)
joblib.dump(lgbm, f'{MODEL_DIR}/lightgbm.pkl')

y_pred_lgbm  = lgbm.predict(X_test_scaled)
y_proba_lgbm = lgbm.predict_proba(X_test_scaled)[:, 1]
evaluate_and_store('LightGBM', y_test, y_pred_lgbm, y_proba_lgbm)

---
## Section 8 — Deep Learning Models (Keras/TensorFlow)

Input shape: `(WINDOW_SIZE=20, n_features=15)`. All DL models use class weights instead of SMOTE (class weighting is the standard DL imbalance strategy). `EarlyStopping` prevents overfitting on CPU.

In [ ]:
def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(history.history['loss'],     label='train loss')
    axes[0].plot(history.history['val_loss'], label='val loss')
    axes[0].set_title(f'{model_name} — Loss')
    axes[0].legend()
    axes[1].plot(history.history['auc'],     label='train AUC')
    axes[1].plot(history.history['val_auc'], label='val AUC')
    axes[1].set_title(f'{model_name} — AUC')
    axes[1].legend()
    plt.tight_layout(); plt.show()


early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
N_FEAT = len(FEATURE_COLS)

In [ ]:
# ── 8a — BiLSTM ───────────────────────────────────────────────────────────────
inputs = keras.Input(shape=(WINDOW_SIZE, N_FEAT))
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Bidirectional(layers.LSTM(32))(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(16, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

bilstm = keras.Model(inputs, outputs, name='BiLSTM')
bilstm.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

hist_bilstm = bilstm.fit(
    X_dl_train, y_dl_train,
    epochs=100, batch_size=256,
    validation_split=0.15,
    class_weight=class_weight_dl,
    callbacks=[early_stop],
    verbose=0
)
bilstm.save(f'{MODEL_DIR}/bilstm.keras')
plot_training_history(hist_bilstm, 'BiLSTM')

y_proba_bilstm = bilstm.predict(X_dl_test, verbose=0).flatten()
y_pred_bilstm  = (y_proba_bilstm >= 0.5).astype(int)
evaluate_and_store('BiLSTM', y_dl_test, y_pred_bilstm, y_proba_bilstm)

In [ ]:
# ── 8b — CNN-LSTM (CLSTM) ────────────────────────────────────────────────────
inputs = keras.Input(shape=(WINDOW_SIZE, N_FEAT))
x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.LSTM(64)(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

cnn_lstm = keras.Model(inputs, outputs, name='CNN_LSTM')
cnn_lstm.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

hist_cnnlstm = cnn_lstm.fit(
    X_dl_train, y_dl_train,
    epochs=100, batch_size=256,
    validation_split=0.15,
    class_weight=class_weight_dl,
    callbacks=[early_stop],
    verbose=0
)
cnn_lstm.save(f'{MODEL_DIR}/cnn_lstm.keras')
plot_training_history(hist_cnnlstm, 'CNN-LSTM')

y_proba_cnnlstm = cnn_lstm.predict(X_dl_test, verbose=0).flatten()
y_pred_cnnlstm  = (y_proba_cnnlstm >= 0.5).astype(int)
evaluate_and_store('CNN-LSTM', y_dl_test, y_pred_cnnlstm, y_proba_cnnlstm)

In [ ]:
# ── 8c — Transformer (lightweight) ───────────────────────────────────────────
# Reduce num_heads=2 if per-epoch training time > 30 min on CPU.
inputs = keras.Input(shape=(WINDOW_SIZE, N_FEAT))
# Project to model dimension
x = layers.Dense(32)(inputs)
# Multi-head self-attention
attn_out = layers.MultiHeadAttention(num_heads=4, key_dim=8)(x, x)
x = layers.Add()([x, attn_out])
x = layers.LayerNormalization()(x)
# Feed-forward block
ff = layers.Dense(64, activation='relu')(x)
ff = layers.Dense(32)(ff)
x = layers.Add()([x, ff])
x = layers.LayerNormalization()(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

transformer = keras.Model(inputs, outputs, name='Transformer')
transformer.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[keras.metrics.AUC(name='auc')]
)

hist_transformer = transformer.fit(
    X_dl_train, y_dl_train,
    epochs=100, batch_size=256,
    validation_split=0.15,
    class_weight=class_weight_dl,
    callbacks=[early_stop],
    verbose=0
)
transformer.save(f'{MODEL_DIR}/transformer.keras')
plot_training_history(hist_transformer, 'Transformer')

y_proba_trans = transformer.predict(X_dl_test, verbose=0).flatten()
y_pred_trans  = (y_proba_trans >= 0.5).astype(int)
evaluate_and_store('Transformer', y_dl_test, y_pred_trans, y_proba_trans)

---
## Section 9 — Save All Results

In [ ]:
# ── Align test targets (ML uses y_test, DL uses y_dl_test) ────────────────────
# NOTE: y_test (ML) and y_dl_test (DL) may differ in length due to sequence
# construction. We store predictions per model with their own y_true column.

ml_pred_df = pd.DataFrame({
    'y_true':                y_test,
    'logistic_regression':   y_pred_lr,
    'logistic_regression_p': y_proba_lr,
    'random_forest':         y_pred_rf,
    'random_forest_p':       y_proba_rf,
    'lightgbm':              y_pred_lgbm,
    'lightgbm_p':            y_proba_lgbm,
})

dl_pred_df = pd.DataFrame({
    'y_true':          y_dl_test,
    'bilstm':          y_pred_bilstm,
    'bilstm_p':        y_proba_bilstm,
    'cnn_lstm':        y_pred_cnnlstm,
    'cnn_lstm_p':      y_proba_cnnlstm,
    'transformer':     y_pred_trans,
    'transformer_p':   y_proba_trans,
})

ml_pred_df.to_csv(f'{RESULTS_DIR}/ml_predictions.csv', index=False)
dl_pred_df.to_csv(f'{RESULTS_DIR}/dl_predictions.csv', index=False)

print('All predictions saved.')
print(f'  ML test set : {ml_pred_df.shape[0]:,} samples')
print(f'  DL test set : {dl_pred_df.shape[0]:,} samples')

In [ ]:
# ── Quick sanity check on all 6 models ───────────────────────────────────────
print('\n=== Quick Sanity Check (threshold=0.5) ===\n')
for name, preds in [
    ('Logistic Regression', (y_test, y_pred_lr)),
    ('Random Forest',       (y_test, y_pred_rf)),
    ('LightGBM',            (y_test, y_pred_lgbm)),
    ('BiLSTM',              (y_dl_test, y_pred_bilstm)),
    ('CNN-LSTM',            (y_dl_test, y_pred_cnnlstm)),
    ('Transformer',         (y_dl_test, y_pred_trans)),
]:
    y_true, y_pred = preds
    cm = confusion_matrix(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    print(f'{name:25s} | MCC={mcc:+.4f} | CM={cm.tolist()}')